# Analyse exploratoire de données postopératoires

## 1. Introduction

Ce projet vise à explorer un jeu de données postopératoires afin de décrire les caractéristiques physiologiques observées, d'évaluer la qualité des données et d'étudier les associations entre certaines variables et la destination du patient après la salle de réveil.

L'analyse porte sur 90 observations et s'inscrit dans une démarche exploratoire. Compte tenu de la petite taille de l'échantillon et du faible nombre de patients transférés aux soins intensifs, les résultats ne permettent pas de tirer des conclusions cliniques généralisables.

In [8]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Paramètres d'affichage
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# Chemin vers les données
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "post-operative.data"


## 2. Importation et définition des variables

Le jeu de données `post-operative.data` ne contient pas de ligne d'en-tête. Les noms des variables ont donc été attribués manuellement à partir de la documentation fournie dans le fichier `post-operative.names`, à la section *Attribute Information*.

Les neuf colonnes correspondent aux huit variables descriptives suivantes : température corporelle interne (`L-CORE`), température de surface (`L-SURF`), saturation en oxygène (`L-O2`), pression artérielle (`L-BP`), stabilité de la température de surface (`SURF-STBL`), stabilité de la température interne (`CORE-STBL`), stabilité de la pression artérielle (`BP-STBL`) et niveau de confort du patient (`COMFORT`). La neuvième variable, `ADM-DECS`, représente la décision concernant la destination du patient après son passage en salle de réveil.

Cette attribution des noms permet de rendre les données plus lisibles et de faciliter leur interprétation lors des analyses statistiques. Les valeurs `?`, utilisées dans le fichier original pour représenter les données manquantes, sont identifiées comme des valeurs manquantes lors de l'importation.

In [9]:
# Noms des variables
colonnes = [
    "L-CORE", "L-SURF", "L-O2", "L-BP",
    "SURF-STBL", "CORE-STBL", "BP-STBL",
    "COMFORT", "ADM-DECS"
]

# Importation
df = pd.read_csv(
    DATA_FILE,
    header=None,
    names=colonnes,
    na_values=["?"],
    skipinitialspace=True
)

# Nettoyage des espaces et conversion numérique
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df["COMFORT"] = pd.to_numeric(df["COMFORT"], errors="coerce")

# Aperçu
print(f"Dimensions : {df.shape[0]} lignes et {df.shape[1]} colonnes")
display(df.head())


Dimensions : 90 lignes et 9 colonnes


,L-CORE,L-SURF,L-O2,L-BP,SURF-STBL,CORE-STBL,BP-STBL,COMFORT,ADM-DECS
0,mid,low,excellent,mid,stable,stable,stable,15.0,A
1,mid,high,excellent,high,stable,stable,stable,10.0,S
2,high,low,excellent,high,stable,stable,mod-stable,10.0,A
3,mid,low,good,high,stable,unstable,mod-stable,15.0,A
4,mid,mid,excellent,high,stable,stable,stable,10.0,A


## 3. Contrôle qualité et exploration initiale

Avant de procéder aux analyses statistiques, une étape de contrôle qualité est réalisée afin de vérifier la structure du jeu de données, les types de variables, la présence de valeurs manquantes et la cohérence des observations.

Le jeu de données contient 90 observations et 9 variables, dont huit variables descriptives et une variable cible correspondant à la destination du patient après son passage en salle de réveil. Les noms des variables ont été attribués à partir de la documentation originale fournie dans le fichier `post-operative.names`.

L'exploration initiale comprend l'examen des dimensions du jeu de données, des types de variables, des statistiques descriptives et de la distribution des catégories. Une attention particulière est accordée aux valeurs manquantes de la variable `COMFORT`, identifiées par le symbole `?` dans le fichier original.

La distribution de la variable cible `ADM-DECS` est également examinée afin de caractériser les différentes destinations des patients et d'identifier un éventuel déséquilibre entre les catégories.

Cette étape permet de documenter la qualité des données et de déterminer les précautions méthodologiques nécessaires avant les analyses statistiques. Les éventuelles anomalies détectées seront examinées avant de décider des traitements à appliquer.

In [10]:

# Dimensions du jeu de données
print("Dimensions :", df.shape)

# Types des variables
print("\nTypes des variables :")
print(df.dtypes)

# Nombre de valeurs manquantes par variable
print("\nValeurs manquantes :")
print(df.isna().sum())

# Vérification des doublons
print("\nNombre de lignes dupliquées :", df.duplicated().sum())

# Répartition de la variable cible
print("\nRépartition des destinations :")
print(df["ADM-DECS"].value_counts(dropna=False))

print("\nPourcentages :")
print(
    (df["ADM-DECS"].value_counts(normalize=True) * 100).round(1)
)


Dimensions : (90, 9)

Types des variables :
L-CORE        object
L-SURF        object
L-O2          object
L-BP          object
SURF-STBL     object
CORE-STBL     object
BP-STBL       object
COMFORT      float64
ADM-DECS      object
dtype: object

Valeurs manquantes :
L-CORE       0
L-SURF       0
L-O2         0
L-BP         0
SURF-STBL    0
CORE-STBL    0
BP-STBL      0
COMFORT      3
ADM-DECS     0
dtype: int64

Nombre de lignes dupliquées : 10

Répartition des destinations :
ADM-DECS
A    64
S    24
I     2
Name: count, dtype: int64

Pourcentages :
ADM-DECS
A    71.1
S    26.7
I     2.2
Name: proportion, dtype: float64


## 4. Analyse descriptive

L'analyse descriptive vise à caractériser les variables du jeu de données postopératoires avant d'étudier leurs associations statistiques. Elle comprend l'examen des distributions des variables catégorielles, la description de la variable numérique `COMFORT` et l'analyse de la distribution de la variable cible `ADM-DECS`.

Cette étape permet de mieux comprendre les caractéristiques de l'échantillon, de repérer les catégories les plus fréquentes et de mettre en évidence le déséquilibre éventuel entre les destinations des patients.

Les trois valeurs manquantes de la variable `COMFORT` seront prises en compte dans l'interprétation des résultats. Aucune conclusion clinique ne sera tirée à partir des seules statistiques descriptives.

In [11]:
# Résumé des variables numériques
df.describe()

,COMFORT
count,87.000000
mean,10.942529
std,2.304477
min,5.000000
25%,10.000000
50%,10.000000
75%,10.000000
max,15.000000


In [12]:
# Résumé de toutes les variables
df.describe(include="all")

,L-CORE,L-SURF,L-O2,L-BP,SURF-STBL,CORE-STBL,BP-STBL,COMFORT,ADM-DECS
count,90,90,90,90,90,90,90,87.000000,90
unique,3,3,2,3,2,3,3,NaN,3
top,mid,mid,good,mid,stable,stable,stable,NaN,A
freq,58,48,47,57,45,83,46,NaN,64
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.942529,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.304477,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,NaN
